# Honest Consultant — Expanded Dataset Analysis

The **honest consultant** is told to determine the correct answer and argue for it.
Compare with the **free choice** consultant (picks strategically) and **assigned** consultants.

- **Dataset:** 350 questions (150 from v2 + 200 from v3)
- **Model:** gpt-4o-mini, 8 candidates per question (temperature 0.8)
- **Judge:** no-context CoT judge (gpt-4o-mini, temperature 0)
- **Script:** `scripts/honest_sweep.py`
- **Results:** `exp/honest_sweep/`

---
## 1. BoN Sweep: Honest vs Free Choice vs Assigned

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

bon_values = [1, 2, 4, 8]

# Load honest results
honest_df = pd.read_csv("../exp/honest_sweep/results.csv")
honest_all = honest_df[honest_df["subset"] == "all"]

# Load free choice / assigned results (combined v2+v3)
v2 = pd.read_csv("../exp/consultant_sweep_v2/results.csv")
v3 = pd.read_csv("../exp/consultant_sweep_v3/results.csv")

def combined_wr(ct, bon):
    v2_row = v2[(v2["consultant_type"] == ct) & (v2["bon"] == bon)].iloc[0]
    v3_row = v3[(v3["consultant_type"] == ct) & (v3["bon"] == bon)].iloc[0]
    return (v2_row["consultant_wins"] * 150 + v3_row["consultant_wins"] * 200) / 350

header = "| Consultant | " + " | ".join(f"BoN={b}" for b in bon_values) + " |"
sep = "|---| " + " | ".join("---" for _ in bon_values) + " |"
rows = [header, sep]

for ct_label, getter in [
    ("Assigned correct", lambda b: combined_wr("assigned_correct", b)),
    ("Assigned incorrect", lambda b: combined_wr("assigned_incorrect", b)),
    ("Free choice", lambda b: combined_wr("free_choice", b)),
    ("**Honest**", lambda b: honest_all[honest_all["bon"] == b].iloc[0]["consultant_wins"]),
]:
    vals = " | ".join(f"{getter(b):.1%}" for b in bon_values)
    rows.append(f"| {ct_label} | {vals} |")

display(Markdown("### Consultant Win Rate (Combined, n=350)\n\n" + "\n".join(rows)))

# Also show judge accuracy and frac chose correct for honest
display(Markdown("### Honest Consultant Details\n"))
header2 = "| BoN | Judge Correct | Consultant Wins | Chose Correct | N |"
sep2 = "|---|---|---|---|---|"
rows2 = [header2, sep2]
for _, r in honest_all.iterrows():
    rows2.append(f"| {int(r['bon'])} | {r['judge_correct']:.1%} | {r['consultant_wins']:.1%} | {r['frac_chose_correct']:.1%} | {int(r['n'])} |")
display(Markdown("\n".join(rows2)))

---
## 2. Honest Consultant Side Selection

In [ ]:
import matplotlib.pyplot as plt

# Load honest details
with open("../exp/honest_sweep/details.json") as f:
    honest_details = json.load(f)

honest_pq = []
total_correct = 0
total_incorrect = 0
total_unknown = 0

for q in honest_details:
    sides = q["sides"]
    n_cor = sum(1 for s in sides if s == "correct")
    n_inc = sum(1 for s in sides if s == "incorrect")
    n_unk = len(sides) - n_cor - n_inc
    total_correct += n_cor
    total_incorrect += n_inc
    total_unknown += n_unk
    honest_pq.append({"idx": q["idx"], "batch": q["batch"],
                       "n_correct": n_cor, "n_incorrect": n_inc, "n_unknown": n_unk})

total = total_correct + total_incorrect + total_unknown
display(Markdown(f"""### Honest Consultant — Raw Side Selection ({total} responses)

| Side | Count | Fraction |
|---|---|---|
| Correct | {total_correct} | {total_correct/total:.1%} |
| Incorrect | {total_incorrect} | {total_incorrect/total:.1%} |
| Unknown | {total_unknown} | {total_unknown/total:.1%} |"""))

# Compare with free choice
fc_correct = 0
fc_incorrect = 0
fc_unknown = 0
for batch_label, cache_path, n_qs in [
    ("v2", Path("../exp/consultant_sweep_v2/cache"), 150),
    ("v3", Path("../exp/consultant_sweep_v3/cache"), 200),
]:
    for idx in range(n_qs):
        gf = cache_path / f"{idx}_free_choice_gen.json"
        if not gf.exists():
            continue
        with open(gf) as f:
            gdata = json.load(f)
        sides = gdata["sides"]
        fc_correct += sum(1 for s in sides if s == "correct")
        fc_incorrect += sum(1 for s in sides if s == "incorrect")
        fc_unknown += sum(1 for s in sides if s not in ("correct", "incorrect"))

fc_total = fc_correct + fc_incorrect + fc_unknown
display(Markdown(f"""### Comparison: Side Selection Rates

| | Honest | Free Choice |
|---|---|---|
| Correct | {total_correct/total:.1%} | {fc_correct/fc_total:.1%} |
| Incorrect | {total_incorrect/total:.1%} | {fc_incorrect/fc_total:.1%} |
| Unknown | {total_unknown/total:.1%} | {fc_unknown/fc_total:.1%} |"""))

In [ ]:
hpq_df = pd.DataFrame(honest_pq)
hpq_df["frac_correct"] = hpq_df["n_correct"] / 8

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(hpq_df["frac_correct"], bins=9, range=(0, 1), edgecolor="black", alpha=0.7)
axes[0].axvline(hpq_df["frac_correct"].mean(), color="red", linestyle="--",
                label=f"Mean: {hpq_df['frac_correct'].mean():.1%}")
axes[0].set_xlabel("Fraction of 8 candidates picking correct")
axes[0].set_ylabel("Number of questions")
axes[0].set_title(f"Honest Consultant: Side Selection Distribution (n={len(hpq_df)})")
axes[0].legend()

all_correct = (hpq_df["n_correct"] == 8).sum()
all_incorrect = (hpq_df["n_incorrect"] >= 7).sum()
axes[1].bar(["All 8 correct", "7-8 incorrect"], [all_correct, all_incorrect],
            color=["#2ecc71", "#e74c3c"], edgecolor="black")
axes[1].set_ylabel("Number of questions")
axes[1].set_title("Questions with Strong Side Preference")
for i, v in enumerate([all_correct, all_incorrect]):
    axes[1].text(i, v + 0.5, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 3. Honest Consultant: Heatmap of Incorrect Side Choice by Assigned Win Rates

In [ ]:
import seaborn as sns

# Build per-question data: honest side choice + assigned correct/incorrect win rates
honest_cache = Path("../exp/honest_sweep/cache")

q_data = []
for batch_label, source_cache, n_qs in [
    ("v2", Path("../exp/consultant_sweep_v2/cache"), 150),
    ("v3", Path("../exp/consultant_sweep_v3/cache"), 200),
]:
    for idx in range(n_qs):
        # Assigned win rates from existing sweeps
        jf_inc = source_cache / f"{idx}_assigned_incorrect_judge.json"
        jf_cor = source_cache / f"{idx}_assigned_correct_judge.json"
        # Honest side choices
        hf = honest_cache / f"{batch_label}_{idx}_honest_gen.json"
        if not all(p.exists() for p in [jf_inc, jf_cor, hf]):
            continue
        with open(jf_inc) as f:
            jdata_inc = json.load(f)
        with open(jf_cor) as f:
            jdata_cor = json.load(f)
        with open(hf) as f:
            hdata = json.load(f)

        inc_wr = sum(1 for c in jdata_inc["is_correct"] if not c) / len(jdata_inc["is_correct"])
        cor_wr = sum(1 for c in jdata_cor["is_correct"] if c) / len(jdata_cor["is_correct"])
        sides = hdata["sides"]
        n_incorrect = sum(1 for s in sides if s == "incorrect")
        n_total = len(sides)

        q_data.append({
            "idx": idx, "batch": batch_label,
            "inc_wr": inc_wr, "cor_wr": cor_wr,
            "n_incorrect": n_incorrect, "n_total": n_total,
        })

qdf = pd.DataFrame(q_data)
print(f"Loaded {len(qdf)} questions")

# Bin assigned correct and incorrect win rates
bins = [0, 0.25, 0.5, 0.75, 1.0]
bin_labels = ["0\u201325%", "25\u201350%", "50\u201375%", "75\u2013100%"]
qdf["cor_wr_bin"] = pd.cut(qdf["cor_wr"], bins=bins, labels=bin_labels, include_lowest=True)
qdf["inc_wr_bin"] = pd.cut(qdf["inc_wr"], bins=bins, labels=bin_labels, include_lowest=True)

# Aggregate: total incorrect responses / total responses per bin
pivot_n_inc = qdf.groupby(["inc_wr_bin", "cor_wr_bin"])["n_incorrect"].sum().unstack(fill_value=0)
pivot_n_tot = qdf.groupby(["inc_wr_bin", "cor_wr_bin"])["n_total"].sum().unstack(fill_value=0)

pivot_n_inc = pivot_n_inc.reindex(index=bin_labels[::-1], columns=bin_labels).fillna(0)
pivot_n_tot = pivot_n_tot.reindex(index=bin_labels[::-1], columns=bin_labels).fillna(0)

pivot_frac = pivot_n_inc / pivot_n_tot

# Annotation: frac + counts
annot = pivot_frac.copy().astype(str)
for r in annot.index:
    for c in annot.columns:
        n_tot = int(pivot_n_tot.loc[r, c])
        n_inc = int(pivot_n_inc.loc[r, c])
        if n_tot == 0:
            annot.loc[r, c] = ""
        else:
            frac = n_inc / n_tot
            annot.loc[r, c] = f"{frac:.0%}\n({n_inc}/{n_tot})"

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    pivot_frac, annot=annot, fmt="", cmap="RdYlGn_r",
    vmin=0, vmax=1, linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Frac Responses Picking Incorrect"},
    ax=ax,
)
ax.set_xlabel("Assigned Correct Win Rate", fontsize=13)
ax.set_ylabel("Assigned Incorrect Win Rate", fontsize=13)
ax.set_title("Honest Consultant: Fraction of Responses Picking Incorrect\nby Assigned Correct/Incorrect Win Rates (n=350)", fontsize=14)
plt.tight_layout()
plt.show()

total_inc = qdf["n_incorrect"].sum()
total_resp = qdf["n_total"].sum()
print(f"Overall: {total_inc}/{total_resp} responses picked incorrect ({total_inc/total_resp:.1%})")

---
## 4. Side-by-Side: Honest vs Free Choice Heatmaps

In [ ]:
# Build free choice data with the same structure
fc_data = []
for batch_label, source_cache, n_qs in [
    ("v2", Path("../exp/consultant_sweep_v2/cache"), 150),
    ("v3", Path("../exp/consultant_sweep_v3/cache"), 200),
]:
    for idx in range(n_qs):
        jf_inc = source_cache / f"{idx}_assigned_incorrect_judge.json"
        jf_cor = source_cache / f"{idx}_assigned_correct_judge.json"
        gf = source_cache / f"{idx}_free_choice_gen.json"
        if not all(p.exists() for p in [jf_inc, jf_cor, gf]):
            continue
        with open(jf_inc) as f:
            jdata_inc = json.load(f)
        with open(jf_cor) as f:
            jdata_cor = json.load(f)
        with open(gf) as f:
            gdata = json.load(f)

        inc_wr = sum(1 for c in jdata_inc["is_correct"] if not c) / len(jdata_inc["is_correct"])
        cor_wr = sum(1 for c in jdata_cor["is_correct"] if c) / len(jdata_cor["is_correct"])
        sides = gdata["sides"]
        n_incorrect = sum(1 for s in sides if s == "incorrect")
        n_total = len(sides)

        fc_data.append({
            "idx": idx, "batch": batch_label,
            "inc_wr": inc_wr, "cor_wr": cor_wr,
            "n_incorrect": n_incorrect, "n_total": n_total,
        })

fc_df = pd.DataFrame(fc_data)

def build_heatmap_data(df):
    bins = [0, 0.25, 0.5, 0.75, 1.0]
    bl = ["0\u201325%", "25\u201350%", "50\u201375%", "75\u2013100%"]
    df = df.copy()
    df["cor_wr_bin"] = pd.cut(df["cor_wr"], bins=bins, labels=bl, include_lowest=True)
    df["inc_wr_bin"] = pd.cut(df["inc_wr"], bins=bins, labels=bl, include_lowest=True)
    p_inc = df.groupby(["inc_wr_bin", "cor_wr_bin"])["n_incorrect"].sum().unstack(fill_value=0)
    p_tot = df.groupby(["inc_wr_bin", "cor_wr_bin"])["n_total"].sum().unstack(fill_value=0)
    p_inc = p_inc.reindex(index=bl[::-1], columns=bl).fillna(0)
    p_tot = p_tot.reindex(index=bl[::-1], columns=bl).fillna(0)
    p_frac = p_inc / p_tot
    annot = p_frac.copy().astype(str)
    for r in annot.index:
        for c in annot.columns:
            nt = int(p_tot.loc[r, c])
            ni = int(p_inc.loc[r, c])
            if nt == 0:
                annot.loc[r, c] = ""
            else:
                annot.loc[r, c] = f"{ni/nt:.0%}\n({ni}/{nt})"
    return p_frac, annot

h_frac, h_annot = build_heatmap_data(qdf)
fc_frac, fc_annot = build_heatmap_data(fc_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(
    fc_frac, annot=fc_annot, fmt="", cmap="RdYlGn_r",
    vmin=0, vmax=1, linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Frac Picking Incorrect"},
    ax=axes[0],
)
axes[0].set_xlabel("Assigned Correct Win Rate", fontsize=12)
axes[0].set_ylabel("Assigned Incorrect Win Rate", fontsize=12)
axes[0].set_title("Free Choice Consultant", fontsize=14)

sns.heatmap(
    h_frac, annot=h_annot, fmt="", cmap="RdYlGn_r",
    vmin=0, vmax=1, linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Frac Picking Incorrect"},
    ax=axes[1],
)
axes[1].set_xlabel("Assigned Correct Win Rate", fontsize=12)
axes[1].set_ylabel("Assigned Incorrect Win Rate", fontsize=12)
axes[1].set_title("Honest Consultant", fontsize=14)

fig.suptitle("Fraction of Responses Picking Incorrect Answer\nby Assigned Correct/Incorrect Win Rates (n=350)",
             fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

fc_total_inc = fc_df["n_incorrect"].sum()
fc_total_resp = fc_df["n_total"].sum()
h_total_inc = qdf["n_incorrect"].sum()
h_total_resp = qdf["n_total"].sum()
print(f"Free Choice overall: {fc_total_inc}/{fc_total_resp} picked incorrect ({fc_total_inc/fc_total_resp:.1%})")
print(f"Honest overall:      {h_total_inc}/{h_total_resp} picked incorrect ({h_total_inc/h_total_resp:.1%})")

---
## 5. Per-Question: Honest vs Free Choice Side Agreement

In [ ]:
# For each question, compare honest and free choice side preferences
comparison = []
for batch_label, source_cache, n_qs in [
    ("v2", Path("../exp/consultant_sweep_v2/cache"), 150),
    ("v3", Path("../exp/consultant_sweep_v3/cache"), 200),
]:
    for idx in range(n_qs):
        hf = honest_cache / f"{batch_label}_{idx}_honest_gen.json"
        gf = source_cache / f"{idx}_free_choice_gen.json"
        if not hf.exists() or not gf.exists():
            continue
        with open(hf) as f:
            hdata = json.load(f)
        with open(gf) as f:
            gdata = json.load(f)

        h_frac_correct = sum(1 for s in hdata["sides"] if s == "correct") / len(hdata["sides"])
        fc_frac_correct = sum(1 for s in gdata["sides"] if s == "correct") / len(gdata["sides"])
        comparison.append({
            "idx": idx, "batch": batch_label,
            "honest_frac_correct": h_frac_correct,
            "fc_frac_correct": fc_frac_correct,
        })

comp_df = pd.DataFrame(comparison)
corr = comp_df["honest_frac_correct"].corr(comp_df["fc_frac_correct"])

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(comp_df["fc_frac_correct"], comp_df["honest_frac_correct"],
           alpha=0.4, edgecolors="black", s=30)
ax.plot([0, 1], [0, 1], "gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Free Choice: Frac Picking Correct", fontsize=12)
ax.set_ylabel("Honest: Frac Picking Correct", fontsize=12)
ax.set_title(f"Per-Question Side Selection: Honest vs Free Choice (r={corr:.2f})", fontsize=13)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.text(0.05, 0.95, f"r = {corr:.2f} (n={len(comp_df)})", transform=ax.transAxes,
        fontsize=12, verticalalignment="top")
plt.tight_layout()
plt.show()

# Where do they disagree?
honest_prefers_correct = comp_df[comp_df["honest_frac_correct"] > comp_df["fc_frac_correct"] + 0.25]
fc_prefers_correct = comp_df[comp_df["fc_frac_correct"] > comp_df["honest_frac_correct"] + 0.25]
print(f"Questions where honest picks correct more often (+25pp): {len(honest_prefers_correct)}")
print(f"Questions where free choice picks correct more often (+25pp): {len(fc_prefers_correct)}")

---
## 6. BoN-Selected Side Choice: Honest vs Free Choice

In [ ]:
bon_values = [1, 2, 4, 8]

# Honest
h_chose_correct = []
h_chose_incorrect = []
for bon in bon_values:
    cc = 0
    ci = 0
    for q in honest_details:
        is_correct = q["is_correct"][:bon]
        sides = q["sides"][:bon]
        best = 0
        for i in range(min(bon, len(is_correct))):
            s = sides[i]
            if (s == "correct" and q["is_correct"][i]) or \
               (s == "incorrect" and not q["is_correct"][i]):
                best = i
                break
        chosen = q["sides"][best]
        if chosen == "correct":
            cc += 1
        elif chosen == "incorrect":
            ci += 1
    h_chose_correct.append(f"{cc/len(honest_details):.1%}")
    h_chose_incorrect.append(f"{ci/len(honest_details):.1%}")

# Free choice (from combined details)
all_fc_details = []
for dp in ["../exp/consultant_sweep_v2/details.json", "../exp/consultant_sweep_v3/details.json"]:
    with open(dp) as f:
        all_fc_details.extend(json.load(f))

fc_chose_correct = []
fc_chose_incorrect = []
for bon in bon_values:
    cc = 0
    ci = 0
    for q in all_fc_details:
        tr = q["types"]["free_choice"]
        is_correct = tr["is_correct"][:bon]
        sides = tr["sides"][:bon]
        best = 0
        for i in range(min(bon, len(is_correct))):
            s = sides[i]
            if (s == "correct" and tr["is_correct"][i]) or \
               (s == "incorrect" and not tr["is_correct"][i]):
                best = i
                break
        chosen = tr["sides"][best]
        if chosen == "correct":
            cc += 1
        elif chosen == "incorrect":
            ci += 1
    fc_chose_correct.append(f"{cc/len(all_fc_details):.1%}")
    fc_chose_incorrect.append(f"{ci/len(all_fc_details):.1%}")

header = "| | " + " | ".join(f"BoN={b}" for b in bon_values) + " |"
sep = "|---| " + " | ".join("---" for _ in bon_values) + " |"
display(Markdown("### BoN-Selected Side Choice\n\n" + "\n".join([
    header, sep,
    "| Honest: Correct | " + " | ".join(h_chose_correct) + " |",
    "| Honest: Incorrect | " + " | ".join(h_chose_incorrect) + " |",
    "| Free Choice: Correct | " + " | ".join(fc_chose_correct) + " |",
    "| Free Choice: Incorrect | " + " | ".join(fc_chose_incorrect) + " |",
])))